In [15]:
import sys
from pathlib import Path
import numpy as np
import torch
import pandas as pd
from chronos import Chronos2Pipeline

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    backtest_21d_rolling,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_CONTEXT_CHRONOS,
    ARTIFACTS_DIR,
    TEST_TICKERS,
)

MODEL_ID = "amazon/chronos-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [16]:
print(f"Loading {MODEL_ID} on {DEVICE}...")
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.float32,
)

def get_forecast_chronos(context: pd.Series, horizon: int):
    """Return list of `horizon` point forecasts (0.5 quantile) from Chronos."""
    try:
        # Chronos requires regular frequency (no gaps). Daily equity data has weekends/holidays,
        # so resample to business-day range and ffill so pd.infer_freq() succeeds.
        context = context.astype(float)
        idx = pd.date_range(context.index.min(), context.index.max(), freq="B")
        context_regular = context.reindex(idx).ffill().bfill().dropna()
        if len(context_regular) < 3:
            return []
        target_values = context_regular.values.flatten()
        timestamps = context_regular.index
        input_df = pd.DataFrame({
            "item_id": ["x"] * len(target_values),
            "timestamp": timestamps,
            "target": target_values,
        })
        forecast_df = pipeline.predict_df(
            input_df,
            prediction_length=horizon,
            quantile_levels=[0.5],
            id_column="item_id",
            timestamp_column="timestamp",
            target="target",
            validate_inputs=True,
        )
        qcol = "0.5" if "0.5" in forecast_df.columns else "predictions"
        if qcol not in forecast_df.columns:
            qcol = forecast_df.columns[-1]
        q = forecast_df[qcol]
        vals = np.asarray(q).ravel()
        return [float(vals[i]) for i in range(min(horizon, len(vals)))]
    except Exception as e:
        print(f"Chronos forecast error: {e}")
        return []

Loading amazon/chronos-2 on cpu...


In [17]:
stacked = load_pool_data()
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL    1255
JNJ     1255
JPM     1255
MSFT    1255
SPY     1255
WMT     1255
dtype: int64


,timestamp,symbol,close
0,2021-03-05,AAPL,121.419998
1,2021-03-08,AAPL,116.360001
2,2021-03-09,AAPL,121.089996
3,2021-03-10,AAPL,119.980003
4,2021-03-11,AAPL,121.959999
5,2021-03-12,AAPL,121.029999
6,2021-03-15,AAPL,123.989998
7,2021-03-16,AAPL,125.570000
8,2021-03-17,AAPL,124.760002
9,2021-03-18,AAPL,120.529999


In [18]:
# Diagnostic: run one Chronos forecast and inspect output (delete or skip after debugging)
_sym = TEST_TICKERS[0]
_grp = stacked[stacked["symbol"] == _sym]
_prices = _grp.set_index("timestamp")["close"].astype(float).dropna().sort_index()
_context = _prices.iloc[-200:]  # last 200 points
_input_df = pd.DataFrame({
    "item_id": ["x"] * len(_context),
    "timestamp": _context.index,
    "target": _context.values,
})
print("Input shape:", _input_df.shape, "| timestamp dtype:", _input_df["timestamp"].dtype)
try:
    _out = pipeline.predict_df(_input_df, prediction_length=21, quantile_levels=[0.5],
                               id_column="item_id", timestamp_column="timestamp", target="target")
    print("Forecast shape:", _out.shape, "| columns:", list(_out.columns))
    print(_out.head(3))
    _got = get_forecast_chronos(_context, 21)
    print("get_forecast_chronos returned len:", len(_got), "| first 3:", _got[:3] if _got else "[]")
except Exception as e:
    print("Error:", type(e).__name__, e)

Input shape: (200, 3) | timestamp dtype: datetime64[s]
Error: ValueError Could not infer frequency for series x


In [19]:
# 21-day-ahead direct forecast; 60-day test window, rolling by 7 days (held-out test assets only)
model_name = "chronos"
all_preds = []
for sym in TEST_TICKERS:
    grp = stacked[stacked["symbol"] == sym]
    if grp.empty:
        continue
    prices = grp.set_index("timestamp")["close"].astype(float).dropna().sort_index()
    if len(prices) < TEST_SIZE + MIN_CONTEXT_CHRONOS:
        continue
    pred = backtest_21d_rolling(
        prices, TEST_SIZE, FORECAST_HORIZON, ROLLING_STEP,
        MIN_CONTEXT_CHRONOS,
        get_forecast_fn=get_forecast_chronos,
    )
    if pred.empty:
        continue
    pred["symbol"] = sym
    all_preds.append(pred)

pred_chronos = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_chronos.groupby("symbol").size() if not pred_chronos.empty else "No predictions (all test symbols skipped or backtest returned empty).")
pred_chronos.head()

c:\capstone_project_unfc\env\Lib\site-packages\chronos\chronos2\dataset.py:89: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  task_target = torch.from_numpy(task_target)


symbol
MSFT    126
SPY     126
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-05,483.160004,480.017395,0,MSFT
1,2025-12-08,491.019989,480.014404,0,MSFT
2,2025-12-09,492.019989,479.870361,0,MSFT
3,2025-12-10,478.559998,479.505127,0,MSFT
4,2025-12-11,483.470001,479.958923,0,MSFT


In [20]:
metrics_rows = []
for sym in pred_chronos["symbol"].unique():
    sub = pred_chronos[pred_chronos["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_chronos)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_chronos_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_chronos_pool.parquet")

     model   symbol        MAE       RMSE    MAPE_%
0  chronos     MSFT  26.474986  30.594880  6.295351
1  chronos      SPY   5.547166   7.194955  0.809967
2  chronos  overall  16.011076  22.469218  3.552659
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_chronos_pool.parquet
